# LiteIE + Zero-DCE — LOLv2 Real, chọn best theo Multinex
Bật GPU + Internet và Add Input dataset LOLv2. Tự tìm Real_captured, bỏ qua Synthetic.
Train toàn bộ Train/Low; mỗi 500 iteration đánh giá Test/Low–Normal. PSNR tăng thì lưu best.pt; SSIM chỉ ghi kèm.
Test cũng được dùng chọn weights, không phải tập đánh giá độc lập.


In [ ]:
DATASET_ROOT = "/kaggle/input"  # Tự tìm duy nhất Real_captured; có thể truyền đường dẫn cụ thể


In [ ]:
from pathlib import Path
import subprocess
import sys
import torch

REPO_URL = "https://github.com/mluu59990-collab/LiteIE.git"
REPO_DIR = Path("/kaggle/working/LiteIE")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    origin = subprocess.check_output(["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True).strip()
    assert origin == REPO_URL, f"Unexpected repository: {origin}"
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
assert (REPO_DIR / "losses.py").exists(), "Checkout chưa có training Zero-DCE"
print("Commit:", subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip())
assert torch.cuda.is_available(), "Hãy bật GPU trong Settings của notebook"
print(torch.__version__, torch.cuda.get_device_name(0))
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements-train.txt")], check=True)


In [ ]:
# Kiểm tra cấu trúc, khả năng đọc ảnh và ghép cặp trước khi train.
command = [sys.executable, "-u", str(REPO_DIR / "train.py"),
           "--dataset-root", DATASET_ROOT, "--variant", "real"]
subprocess.run(command + ["--check-data"], cwd=REPO_DIR, check=True)


In [ ]:
# Zero-DCE defaults: 200 epoch, batch 8, resize 256, Adam lr=1e-4,
# weight decay=1e-4, gradient clip=0.1, không crop/flip.
# Tự resume run cùng recipe. Không dùng last.pt từ recipe cũ.
OUTPUT_DIR = Path("/kaggle/working/liteie-real-best-output")
RESUME = OUTPUT_DIR / "last.pt"
# Sau khi đổi session, Add Input output cũ rồi đặt RESUME tới last.pt đó.
train_command = command + ["--output-dir", str(OUTPUT_DIR)]
if RESUME.exists():
    train_command += ["--resume", str(RESUME)]
subprocess.run(train_command, cwd=REPO_DIR, check=True)


In [ ]:
import json
from IPython.display import FileLink, display
metrics = json.loads((OUTPUT_DIR / "test_metrics.json").read_text())
print(json.dumps(metrics, indent=2))
print("Tải best.pt để dùng inference; last.pt chỉ phục vụ resume.")
display(FileLink(str(OUTPUT_DIR / "best.pt")))


## File cần tải: best.pt
- best.pt: state dict LiteIE ở iteration có PSNR Test cao nhất trong các lần đánh giá.
- evaluations.json: PSNR, SSIM và trạng thái cập nhật best mỗi 500 iteration.
- Train/Normal không cần có. Không tách validation. Loss chỉ dùng Train/Low.
- last.pt chỉ phục vụ resume; không cần tải nếu chỉ muốn inference.
- Output: /kaggle/working/liteie-real-best-output/best.pt.
- Bắt đầu run mới, không resume checkpoint recipe cũ.
